# Chapter 04-05 · Leakage lab: four kinds, ranked by damage

**Label:** Core  |  **Time:** ~60 minutes  |  **Difficulty:** moderate

**Prerequisites:** 04-01 for the wall, 04-03 for held-out data, 04-04 for grouped and chronological
splits.

**Position in the learning path:** module 04, chapter 5 of 8.

---

## Why this matters

**Leakage** is information reaching the model that will not be available when it is used for real. It is
the single most common way a machine learning project fails, and it fails in the worst possible manner:
**everything looks excellent right up to the moment it goes live.**

You have met three kinds already, one per chapter. This chapter puts all four in one lab, measures each
on the same footing, and ranks them - because the ranking is not what the folklore says. The step almost
every tutorial warns about turns out to cost **+0.0005**, and a step most people have never been warned
about produces **79.5% accuracy at predicting a coin flip.**

## What you will be able to do

- Name the four kinds of leakage and give the mechanism of each
- Measure the damage each does, rather than assuming
- Explain why fitting a scaler on all your data is nearly harmless and fitting a feature *selector* on it
  is catastrophic - from one principle, not from a list of rules
- Detect leakage from three symptoms, before deployment finds it for you
- State the single question that catches all four

## Warm-up: retrieve, do not reread

1. In 04-01, what made `months_on_file` unusable?
2. In 04-04, why did k-nearest-neighbours suffer more from a random split than logistic regression?
3. What did 04-03 say a suspiciously good score means?

<br>

*Answers: (1) its value was only knowable after the cancellation it was meant to predict. (2) kNN
predicts by looking up similar training rows, and the most similar rows were the same member's other
months. (3) that the answer is in the features - it is a bug report, not a result.*

## One question, four answers

Every kind of leakage is a different answer to the same question:

> **Did anything the model used know something it could not have known at prediction time?**

The four answers are four places that knowledge can come from.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

fig, axes = plt.subplots(2, 2, figsize=(12.5, 6.6))
panels = [
    ("1. Target leakage", "#f6d3bd", "a feature is a disguised copy of the answer",
     "`months_on_file` is the cancellation month.\nMeasured in 04-01: AUC 0.636 -> 1.000"),
    ("2. Duplicate / group leakage", "#cfe3f3", "the same entity is on both sides of the split",
     "the model recognises a member it studied.\nMeasured in 04-04: +0.1052 for kNN"),
    ("3. Temporal leakage", "#cfe8dc", "the model trains on the future",
     "learning from month 17 to predict month 4.\nMeasured in 04-04: +0.0456"),
    ("4. Preprocessing leakage", "#f3d6e3", "a fitted step saw the test rows",
     "a scaler, an imputer, an encoder, or a\nfeature selector, fitted before the split"),
]
for ax, (title, colour, mechanism, evidence) in zip(axes.ravel(), panels):
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, facecolor=colour, edgecolor="white", linewidth=4))
    ax.text(0.5, 0.78, title, ha="center", fontsize=13, fontweight="bold")
    ax.text(0.5, 0.55, mechanism, ha="center", fontsize=10.5, style="italic", color="#333333")
    ax.text(0.5, 0.26, evidence, ha="center", fontsize=9, color="#555555")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xticks([])
    ax.set_yticks([])
    for side in ax.spines.values():
        side.set_visible(False)
fig.suptitle("Four ways the answer gets into the model", fontsize=14, y=0.99)
plt.tight_layout()
plt.show()

The first three have had a chapter each. **The fourth is the subject of this one**, because it is the
only kind that happens in code that looks like tidying up - and because its severity varies by a factor
of five hundred depending on which step you fit.

## The fourth kind: a step that was fitted on everything

Here is the shape of the mistake, and it is completely ordinary:

```python
scaler = StandardScaler().fit(X)          # <- all of X, before the split
X_scaled = scaler.transform(X)
train_X, test_X, train_y, test_y = train_test_split(X_scaled, y)
```

The scaler computed a mean and a standard deviation using the test rows. Every tutorial that mentions
leakage mentions this, usually first.

**So measure it.** Four hundred rows, three columns, a fifth of the values missing, mean-imputed and
standardised - once fitted on everything, once fitted on the training fold only.

**Predict before running:** how much is it worth?

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler


# SYNTHETIC: one informative column, two noise columns, 20% of values missing.
def make_data(rows, seed=5):
    rng = np.random.default_rng(seed)
    signal = rng.normal(size=rows)
    features = np.column_stack([signal + rng.normal(0, 1, rows),
                                rng.normal(size=rows), rng.normal(size=rows)])
    target = (signal + rng.normal(0, 0.8, rows) > 0).astype(int)
    features[rng.random(features.shape) < 0.2] = np.nan
    return features, target


def preprocessing_experiment(features, target, fit_on_everything):
    scores = []
    for train_rows, test_rows in StratifiedKFold(5, shuffle=True, random_state=0).split(features, target):
        source = features if fit_on_everything else features[train_rows]
        imputer = SimpleImputer().fit(source)
        scaler = StandardScaler().fit(imputer.transform(source))

        prepare = lambda part: scaler.transform(imputer.transform(part))
        model = LogisticRegression(max_iter=2000).fit(prepare(features[train_rows]), target[train_rows])
        scores.append(roc_auc_score(target[test_rows], model.predict_proba(prepare(features[test_rows]))[:, 1]))
    return float(np.mean(scores))


rows = []
for size in [40, 100, 400, 2000]:
    features, target = make_data(size)
    leaky = preprocessing_experiment(features, target, True)
    honest = preprocessing_experiment(features, target, False)
    rows.append({"rows": size, "fitted on everything": round(leaky, 4),
                 "fitted on training only": round(honest, 4),
                 "inflation": round(leaky - honest, 4)})
print(pd.DataFrame(rows).to_string(index=False))

**Essentially nothing** - and at three of the four sizes the "leaky" version is very slightly *worse*.

One dataset is one draw, so before concluding anything, repeat it.

In [ ]:
differences = []
for seed in range(30):
    features, target = make_data(100, seed=seed)
    differences.append(preprocessing_experiment(features, target, True)
                       - preprocessing_experiment(features, target, False))
differences = np.array(differences)

print("30 datasets of 100 rows each:")
print("  mean inflation %+.4f" % differences.mean())
print("  standard deviation %.4f" % differences.std())
print("  range %+.4f to %+.4f" % (differences.min(), differences.max()))
print("  the leaky version wins in %.0f%% of the 30 datasets" % (100 * np.mean(differences > 0)))

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.hist(differences, bins=14, color="#cfe3f3", edgecolor="#0072B2")
ax.axvline(0, color="#000000", linewidth=2)
ax.axvline(differences.mean(), color="#D55E00", linestyle="--", linewidth=2)
ax.text(differences.mean() + 0.0008, ax.get_ylim()[1] * 0.8, "mean %+.4f" % differences.mean(),
        color="#D55E00", fontsize=9)
ax.set_xlabel("AUC when fitted on everything, minus AUC when fitted on training only")
ax.set_ylabel("datasets")
ax.set_title("Scaler and imputer leakage: centred on zero, and on the wrong side half the time",
             fontsize=11)
plt.tight_layout()
plt.show()

**Mean +0.0005, and the leaky version wins in only 37% of datasets.** It is not a small effect - it is
**not detectable at all** at this sample size. The distribution straddles zero.

This is worth saying plainly because it contradicts a great deal of received advice: **fitting a scaler
or a mean-imputer on your whole dataset is, on its own, close to harmless.**

Now do the same experiment with a different preprocessing step.

## The same mistake, a different step: 79.5% accuracy on a coin flip

**200 rows. 5,000 columns of pure random noise. A target that is a coin flip.** There is nothing to
learn - by construction, the honest accuracy is 50%.

The only change from the previous experiment is *which* step gets fitted before the split. Here it is a
**feature selector**: keep the 20 columns most associated with the target.

**Predict before running:** what accuracy will the leaky version report?

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

noise_rng = np.random.default_rng(0)
n_rows, n_columns = 200, 5000
noise = noise_rng.normal(size=(n_rows, n_columns))
coin_flip = noise_rng.integers(0, 2, n_rows)

folds = StratifiedKFold(5, shuffle=True, random_state=0)

# WRONG: choose the 20 best columns using every row, then cross-validate on those columns
chosen = SelectKBest(f_classif, k=20).fit(noise, coin_flip)
leaky_scores = cross_val_score(LogisticRegression(max_iter=2000), chosen.transform(noise),
                               coin_flip, cv=folds, scoring="accuracy")

# RIGHT: the selector is part of the model, so it is refitted inside every fold
honest_scores = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20),
                                              LogisticRegression(max_iter=2000)),
                                noise, coin_flip, cv=folds, scoring="accuracy")

print("predicting a coin flip from 5,000 columns of noise")
print("  select first, then cross-validate : %.4f   folds %s"
      % (leaky_scores.mean(), np.round(leaky_scores, 3)))
print("  selection inside the pipeline     : %.4f   folds %s"
      % (honest_scores.mean(), np.round(honest_scores, 3)))
print("  the truth, by construction        : 0.5000")

**79.5% accuracy at predicting a coin flip.** Every fold agrees - 0.85, 0.775, 0.775, 0.75, 0.825 - so
the result looks stable as well as good. Cross-validation did not catch it, because cross-validation
faithfully repeated a procedure that had already been corrupted before it started.

The honest version scores 0.5450, which is 0.5 plus the noise you would expect from 200 rows.

**Why this one is devastating and the scaler was not:**

- A scaler learns **two numbers per column** from the data: a mean and a standard deviation. Those two
  numbers say almost nothing about any individual row's label. The information leaked is real and
  vanishingly small.
- A selector learns **which columns happen to correlate with the target in this particular sample**. That
  is a statement about the target, extracted from every row including the test ones. With 5,000 candidate
  columns and 200 rows, some columns correlate strongly *by chance* - and the selector's whole job is to
  find exactly those.

> **Leakage severity is proportional to how much information about the target the fitted step absorbed.**

That single principle replaces the list of rules. You do not need to remember which steps are dangerous;
you need to ask **"did this step look at `y`?"** A scaler does not. A selector does. So does an encoder
that uses the target, and so does anything whose output changes when you change the labels.

How bad it gets depends on how much room the selector had to search.

In [ ]:
rows = []
for width in [50, 200, 1000, 5000]:
    columns = noise_rng.normal(size=(n_rows, width))
    labels = noise_rng.integers(0, 2, n_rows)
    keep = min(20, width)
    picked = SelectKBest(f_classif, k=keep).fit(columns, labels)
    leaky = cross_val_score(LogisticRegression(max_iter=2000), picked.transform(columns),
                            labels, cv=folds, scoring="accuracy").mean()
    honest = cross_val_score(make_pipeline(SelectKBest(f_classif, k=keep),
                                           LogisticRegression(max_iter=2000)),
                             columns, labels, cv=folds, scoring="accuracy").mean()
    rows.append({"columns to choose from": width, "leaky accuracy": round(leaky, 4),
                 "honest accuracy": round(honest, 4), "inflation": round(leaky - honest, 4)})
width_table = pd.DataFrame(rows)
print(width_table.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.3))
ax.plot(width_table["columns to choose from"], width_table["leaky accuracy"], "o-",
        color="#D55E00", label="selected before splitting")
ax.plot(width_table["columns to choose from"], width_table["honest accuracy"], "s-",
        color="#0072B2", label="selected inside the pipeline")
ax.axhline(0.5, color="#000000", linestyle=":", linewidth=1.4, label="the truth (a coin flip)")
ax.set_xscale("log")
ax.set_xlabel("number of noise columns to choose from (log scale)")
ax.set_ylabel("reported accuracy")
ax.set_ylim(0.35, 0.9)
ax.set_title("The more columns you search, the more convincing the nonsense", fontsize=11)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
correlations = np.array([np.corrcoef(noise[:, j], coin_flip)[0, 1] for j in range(n_columns)])
picked_columns = SelectKBest(f_classif, k=20).fit(noise, coin_flip).get_support()

fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.3))

left.hist(correlations, bins=60, color="#cfe3f3", edgecolor="#0072B2", linewidth=0.4)
for edge in [correlations[picked_columns].min(), correlations[picked_columns].max()]:
    left.axvline(edge, color="#D55E00", linewidth=2)
left.axvline(0, color="#000000", linestyle=":", linewidth=1.2)
left.set_xlabel("correlation with the target")
left.set_ylabel("columns")
left.set_title("All 5,000 columns. True correlation: zero", fontsize=11)
left.text(0.02, 0.94, "the selector keeps\nonly the two tails",
          transform=left.transAxes, color="#D55E00", fontsize=9, va="top")

right.hist(np.abs(correlations), bins=60, color="#dddddd", edgecolor="#999999", linewidth=0.4,
           label="not selected")
right.hist(np.abs(correlations[picked_columns]), bins=60, color="#D55E00", label="the 20 kept")
right.set_xlabel("size of the correlation, ignoring sign")
right.set_ylabel("columns")
right.set_title("The selector's job is to find the luckiest tail", fontsize=11)
right.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("largest correlation among %d noise columns : %.4f" % (n_columns, np.abs(correlations).max()))
print("the 20 kept range from %.4f to %.4f in size"
      % (np.abs(correlations[picked_columns]).min(), np.abs(correlations[picked_columns]).max()))
print("expected largest, from 3.6 / sqrt(200)      : %.4f" % (3.6 / np.sqrt(200)))

The left panel is the dataset's honest state: 5,000 correlations scattered around zero, exactly as random
numbers should be. **Nothing is wrong with the data.**

The right panel is what selection does to it. The orange bars are the columns kept - **the extreme tail,
by construction**, because "keep the 20 most associated columns" is a request for the tail.

**Those correlations are real in this sample and zero in the population.** A model handed only the tail
reproduces them on any subset of these 200 rows - including the test fold, because the tail was chosen
using it.

**The blue line sits on the truth wherever it is put. The orange line climbs with the width of the
search** - 0.525 at 50 columns, 0.825 at 5,000.

This is 04-03's selection premium in a new costume. There, choosing the best of many *models* on a
validation set inflated the score; here, choosing the best of many *columns* on the whole dataset does the
same thing, and for the identical reason: **taking the maximum of many noisy quantities selects the
noise.** It is also 02-07's multiple-comparisons warning, arriving for the third time.

Wide data - genomics, sensor arrays, text with large vocabularies, anything with more columns than rows -
is where this is fatal, and it is exactly the setting where feature selection feels most necessary.

## The same mistake again: encoding a category with the target

One more, because it is the most common version in ordinary tabular work.

**Target encoding** replaces a categorical column with the mean of the target within each category. It is
a genuinely useful technique. It is also a step that looks directly at `y`.

600 rows, a category column with 149 distinct values - an id-like column, a postcode, a product code -
and, once again, **a coin-flip target.**

In [ ]:
encode_rng = np.random.default_rng(3)
n_rows = 600
category = encode_rng.integers(0, 150, n_rows)
label = encode_rng.integers(0, 2, n_rows)

# WRONG: encode using the target of every row, then cross-validate
leaky_encoding = pd.Series(label).groupby(category).mean().reindex(category).to_numpy().reshape(-1, 1)
leaky = cross_val_score(LogisticRegression(max_iter=2000), leaky_encoding, label,
                        cv=folds, scoring="roc_auc").mean()

# RIGHT: encode inside each fold, using training targets only
honest_folds = []
for train_rows, test_rows in folds.split(category.reshape(-1, 1), label):
    means = pd.Series(label[train_rows]).groupby(category[train_rows]).mean()
    fallback = label[train_rows].mean()
    encode = lambda part: pd.Series(category[part]).map(means).fillna(fallback).to_numpy().reshape(-1, 1)
    model = LogisticRegression(max_iter=2000).fit(encode(train_rows), label[train_rows])
    honest_folds.append(roc_auc_score(label[test_rows], model.predict_proba(encode(test_rows))[:, 1]))

print("%d rows, %d categories, target is a coin flip" % (n_rows, len(np.unique(category))))
print("  encoded using every row's target : AUC %.4f" % leaky)
print("  encoded inside each fold         : AUC %.4f" % np.mean(honest_folds))
print("  the truth                        : AUC 0.5000")

In [ ]:
# pick a category whose four rows are NOT all the same label, so the average is informative
sizes = pd.Series(category).value_counts()
for candidate in sizes[sizes >= 4].index:
    example = np.where(category == candidate)[0][:4]
    if 0 < label[example].sum() < 4:
        break
labels_here = label[example]
highlighted = 0

fig, ax = plt.subplots(figsize=(10, 3.9))
for position, (row_index, row_label) in enumerate(zip(example, labels_here)):
    is_focus = position == highlighted
    ax.add_patch(plt.Rectangle((0.3, position - 0.34), 1.9, 0.68,
                               facecolor="#f6d3bd" if is_focus else "#cfe3f3",
                               edgecolor="white", linewidth=2))
    ax.text(1.25, position, "row %d,   y = %d" % (row_index, row_label), ha="center", va="center",
            fontsize=10, fontweight="bold" if is_focus else "normal")
    ax.annotate("", xy=(3.55, 1.5), xytext=(2.25, position),
                arrowprops=dict(arrowstyle="-|>", color="#D55E00" if is_focus else "#999999",
                                linewidth=2.4 if is_focus else 1.2))

ax.add_patch(plt.Rectangle((3.6, 1.12), 2.7, 0.78, facecolor="#f6d3bd", edgecolor="white", linewidth=2))
ax.text(4.95, 1.51, "encoded value =\nmean(%s) = %.2f" % (", ".join(str(v) for v in labels_here),
                                                          labels_here.mean()),
        ha="center", va="center", fontsize=10)

# the round trip: the encoded value is then handed back to the row that helped make it
ax.annotate("", xy=(2.3, highlighted), xytext=(3.55, 1.12),
            arrowprops=dict(arrowstyle="-|>", color="#D55E00", linewidth=2.4,
                            connectionstyle="arc3,rad=0.35"))
ax.text(6.5, 1.5, "row %d contributed one of the\nfour labels, and is then handed\nthe average back as its feature"
        % example[highlighted], ha="left", va="center", fontsize=9.5, color="#D55E00")

ax.set_xlim(0, 10.6)
ax.set_ylim(-1.0, 3.5)
ax.set_xticks([])
ax.set_yticks([])
for side in ax.spines.values():
    side.set_visible(False)
ax.set_title("Why target encoding leaks: a row helps compute its own feature", fontsize=12)
plt.tight_layout()
plt.show()

**AUC 0.7659 from a column that contains no information whatsoever.**

The mechanism is worth spelling out because it explains why high-cardinality columns are the dangerous
ones. With 149 categories over 600 rows, each category holds about four rows. The encoded value for a
category is therefore **the average of about four labels - and each row's own label is one of the four.**
Every row is being handed a quarter of its own answer, wearing a different name.

Done inside the folds it scores 0.4560 - noise around 0.5, as it should.

**And note that this leak is invisible to every check in 04-04.** The split can be perfectly grouped and
perfectly chronological; the corruption happened in the feature-building step, before splitting was ever
considered.

## Duplicate rows: the third kind, measured directly

04-04 covered repeated *entities*. Literal duplicated rows are the extreme case, and they arrive by
routes nobody chooses: a join that fans out, an export run twice, a customer with two ids, the same
article syndicated to three sites.

300 rows, of which 90 are exact copies of another row.

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import KNeighborsClassifier

duplicate_rng = np.random.default_rng(7)
base_rows = 300
signal = duplicate_rng.normal(size=base_rows)
originals = np.column_stack([signal + duplicate_rng.normal(0, 1, base_rows),
                             duplicate_rng.normal(size=base_rows)])
labels = (signal + duplicate_rng.normal(0, 0.8, base_rows) > 0).astype(int)

copied = duplicate_rng.choice(base_rows, size=90, replace=False)
with_duplicates = np.vstack([originals, originals[copied]])
duplicate_labels = np.concatenate([labels, labels[copied]])
original_row_id = np.concatenate([np.arange(base_rows), copied])


def neighbour_score(splitter, **split_args):
    scores = []
    for train_rows, test_rows in splitter.split(with_duplicates, duplicate_labels, **split_args):
        model = KNeighborsClassifier(3).fit(with_duplicates[train_rows], duplicate_labels[train_rows])
        scores.append(roc_auc_score(duplicate_labels[test_rows],
                                    model.predict_proba(with_duplicates[test_rows])[:, 1]))
    return float(np.mean(scores))


print("%d rows, of which %d are exact copies" % (len(with_duplicates), len(copied)))
print("  random split (copies straddle it)  : AUC %.4f"
      % neighbour_score(StratifiedKFold(5, shuffle=True, random_state=0)))
print("  grouped by original row            : AUC %.4f"
      % neighbour_score(GroupKFold(5), groups=original_row_id))

**0.7257 against 0.5931 - an inflation of 0.1326** from nothing more than 90 duplicated rows.

A duplicate split across the boundary is the purest possible leak: the model is given the test row, with
its label, as a training example. A nearest-neighbour model then finds it at distance zero.

**Duplicates are worth checking for on every dataset, before anything else**, and the check is one line:

```python
frame.duplicated().sum()
```

If the answer is not zero, find out why before deciding what to do - 02-04 covered the deciding. What
matters here is that dropping them *after* splitting is too late.

## The ranking

Every leak in module 04, measured the same way: reported score minus honest score.

In [ ]:
measured = pd.DataFrame([
    {"leak": "target leakage\n(a feature is the answer)", "inflation": 0.3641, "chapter": "04-01"},
    {"leak": "target encoding\n(high-cardinality column)", "inflation": float(leaky - np.mean(honest_folds)), "chapter": "04-05"},
    {"leak": "feature selection\n(5,000 columns)", "inflation": float(leaky_scores.mean() - honest_scores.mean()), "chapter": "04-05"},
    {"leak": "duplicate rows\n(30% copied)", "inflation": 0.1326, "chapter": "04-05"},
    {"leak": "group leakage\n(kNN, repeated members)", "inflation": 0.1052, "chapter": "04-04"},
    {"leak": "temporal leakage\n(training on the future)", "inflation": 0.0456, "chapter": "04-04"},
    {"leak": "target overlap\n(windows share months)", "inflation": 0.0368, "chapter": "04-04"},
    {"leak": "scaler and imputer\n(fitted on everything)", "inflation": 0.0005, "chapter": "04-05"},
]).sort_values("inflation")

fig, ax = plt.subplots(figsize=(9.5, 5.4))
colours = ["#0072B2" if value < 0.02 else "#e8a33d" if value < 0.12 else "#D55E00"
           for value in measured.inflation]
bars = ax.barh(range(len(measured)), measured.inflation, color=colours)
for position, (value, chapter) in enumerate(zip(measured.inflation, measured.chapter)):
    ax.text(value + 0.006, position, "%+.4f   (%s)" % (value, chapter), va="center", fontsize=9)
ax.set_yticks(range(len(measured)))
ax.set_yticklabels(measured.leak, fontsize=8.5)
ax.set_xlim(0, 0.47)
ax.set_xlabel("inflation: reported score minus honest score")
ax.set_title("Every leak in module 04, measured on the same scale", fontsize=12)
plt.tight_layout()
plt.show()

print(measured[["leak", "inflation"]].assign(
    leak=measured.leak.str.replace("\n", " "), inflation=measured.inflation.round(4)
).to_string(index=False))

**A factor of seven hundred between the top and the bottom of that chart**, and the ordering is not the
one folklore would give you.

Read it with the severity principle and it becomes predictable rather than arbitrary:

| Leak | Did the step see `y`? | Damage |
|---|---|---|
| Target leakage | the feature **is** `y` | catastrophic |
| Target encoding | yes, directly, per category | severe |
| Feature selection | yes, to rank columns | severe, and grows with the search |
| Duplicate rows | the test row **is** a training row | severe |
| Group leakage | indirectly, through the entity | moderate, and model-dependent |
| Temporal leakage | indirectly, through time | moderate |
| Scaler, imputer | **no** - only column statistics of `X` | negligible |

**The bottom row is the one to internalise.** A scaler, a plain imputer, a one-hot encoder and a
log transform are all functions of `X` alone. They cannot leak the target because they never see it. The
steps that hurt are the ones that consult `y`, plus the ones that let a test row be its own training
example.

### Two important caveats, so the lesson is not over-learned

**Still put the scaler in the pipeline.** The measured cost is 0.0005 and the correct practice is
unchanged - not because of this number, but because (a) some transforms *are* sensitive to extreme values
in ways a mean is not, (b) the discipline is what makes the dangerous cases automatic, and (c) a pipeline
costs nothing. **The argument for correctness here is cheapness, not fear**, and knowing which it is
makes you better at triage when something breaks.

**And do not read "negligible" as "always negligible."** The measurement above is for a mean-imputer and
a standardiser on 100 to 2,000 rows. An imputer that fills with a *model* fitted on all the data is a
different animal, and so is any transform whose fitted parameters could encode individual rows. The
question is always the same one: how much did this step learn, and from what?

## Detecting it before production does

Leakage has three symptoms. None is proof; all three are cheap.

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 5.6))

steps = [
    (0.5, 4.3, "#f6d3bd", "Is the score better than the problem allows?",
     "AUC 1.000 on behavioural churn.\n79.5% on a coin flip."),
    (0.5, 3.0, "#f6d3bd", "Does one feature dominate everything?",
     "Drop the strongest feature.\nIf the model collapses, interrogate that column."),
    (0.5, 1.7, "#f6d3bd", "Do the folds agree suspiciously well?",
     "A leaky procedure is consistently wrong.\nHigh mean and low variance together."),
    (0.5, 0.4, "#cfe8dc", "Then ask, of every feature and every fitted step:",
     "Would this value have been known at prediction time?\nDid this step look at y?"),
]
for x, y_position, colour, title, detail in steps:
    ax.add_patch(plt.Rectangle((x, y_position), 8.0, 1.0, facecolor=colour, edgecolor="white",
                               linewidth=3))
    ax.text(x + 0.25, y_position + 0.66, title, fontsize=11.5, fontweight="bold", va="center")
    ax.text(x + 0.25, y_position + 0.28, detail, fontsize=9, color="#444444", va="center")
for y_position in [4.3, 3.0, 1.7]:
    ax.annotate("", xy=(4.5, y_position - 0.28), xytext=(4.5, y_position),
                arrowprops=dict(arrowstyle="-|>", color="#666666", linewidth=2))
ax.set_xlim(0, 9)
ax.set_ylim(0, 5.6)
ax.set_xticks([])
ax.set_yticks([])
for side in ax.spines.values():
    side.set_visible(False)
ax.set_title("The leakage checklist, in the order that costs least", fontsize=13)
plt.tight_layout()
plt.show()

**1. Is the score better than the problem allows?** The cheapest check, and the one that caught two of
this chapter's four leaks by itself. It requires you to have an opinion about what "good" would be
*before* seeing the number - which is why 04-01 put "what does good look like?" in the framing contract.

**2. Does one feature dominate?** Drop the strongest feature and refit. A model that collapses without a
single column is telling you that column is doing something unusual - sometimes it is genuinely the
signal, and sometimes it is the answer. Either way you now know where to look.

**3. Do the folds agree too well?** 04-04's leaky random-row folds had a standard deviation of 0.0206
while being 0.096 too high; the honest grouped folds had 0.0317. **Leakage raises the mean and lowers the
variance**, because a systematic advantage is present in every fold equally. Tight folds around a
surprising mean is the classic signature.

Then the two questions that actually settle it, applied to every column and every fitted step.

### The structural fix

All of the above is detection. **Prevention is a pipeline**, and it is the subject of 04-07: put every
fitted step - imputer, scaler, encoder, selector - inside a `Pipeline`, and pass the pipeline to the
cross-validator. Then every step is refitted inside every fold automatically, and preprocessing leakage
becomes impossible rather than merely discouraged.

Add the two assertions from 04-04, and the four kinds are covered:

```python
assert frame.duplicated().sum() == 0
assert set(train.entity_id) & set(test.entity_id) == set()
assert train.date.max() < test.date.min()
# and every fitted step lives inside the Pipeline, not before the split
```

## Common misconceptions

**"Cross-validation protects against leakage."**
It repeats whatever procedure you hand it. The coin-flip demonstration was cross-validated, five-fold,
and reported 79.5% with tight folds.

**"Leakage means somebody made a mistake with the split."**
Two of this chapter's four kinds happen before splitting is even considered, in feature-building code.

**"Fitting the scaler on all the data is the classic leak."**
It is the classic *warning*. Measured, it is +0.0005 and lands on the wrong side of zero 63% of the time.
The classic *leak* is a step that looked at the target.

**"A tiny leak is still a leak, so all leaks matter equally."**
They differ by a factor of seven hundred here. Treating them equally means spending your attention
uniformly on things that are not uniformly important - and the loudest advice points at the smallest one.

**"If my score is realistic, there is no leakage."**
Leakage inflates relative to the truth, not relative to your expectations. A genuinely hard problem with
a serious leak can still report a modest-looking number.

**"I dropped the duplicates, so that is handled."**
Only if you dropped them before splitting. And near-duplicates - the same entity, not the same row - are
not caught by `duplicated()` at all.

**"Feature selection has to be done on all the data or you get different features per fold."**
Different features per fold is *correct*. The variation is information: if the selected set changes
completely between folds, the selection is not finding anything stable, and that is what you needed to
know.

## Exercises

Solutions: `solutions/04_workflow/04-05_leakage_solutions.ipynb`.

### Quick understanding

**E1.** Name the four kinds of leakage and, in one clause each, what information moves and where from.

**E2.** State the severity principle in one sentence, and use it to predict whether a `PolynomialFeatures`
transform fitted before the split is dangerous.

**E3.** Why did cross-validation fail to catch the coin-flip result?

### Hand calculation

**E4.** A target-encoded column has 500 rows and 250 categories, so about 2 rows per category. What
fraction of a row's encoded value is its own label? Repeat for 25 categories (20 rows each). State the
rule this gives you about which categorical columns are risky.

**E5.** You have 200 rows and 5,000 noise columns and a coin-flip target. For a single column, the
correlation with the target has a standard deviation of about `1/sqrt(200)` = 0.071. Roughly how large
would you expect the **largest** of 5,000 such correlations to be? (The expected maximum of 5,000
standard normals is about 3.6.) Comment on what a selector will therefore find.

**E6.** A model scores 0.96 with 5-fold standard deviation 0.004 on a problem where the best published
result is 0.83. Give the three symptoms from the checklist and say which two are triggered.

**E7.** In the duplicate experiment, 90 of 390 rows are copies. Under a random 80/20 split, compute the
expected number of test rows whose twin is in the training set. (Each copy and its original are assigned
independently.)

### Coding

**E8.** Write `leakage_report(frame, entity_column, time_column)` that checks for duplicate rows, prints
the number of entities appearing more than once, and reports the overlap of time ranges. Run it on
04-04's stacked member-month table.

**E9.** Reproduce the coin-flip demonstration with `k` = 1, 5, 20 and 100 selected columns. Does keeping
*more* columns make the leak better or worse? Explain the shape.

**E10.** Take the target-encoding experiment and add **smoothing**: blend each category mean towards the
overall mean with a weight that depends on the category's size. Does smoothing fix the leak when the
encoding is still computed on all rows? Report the number and say what that proves about where the
problem lies.

**E11.** Build a `Pipeline` containing `SimpleImputer`, `StandardScaler`, `SelectKBest` and
`LogisticRegression`, and cross-validate it on the noise data. Confirm it returns to chance. Then break
it deliberately by moving one step outside, and confirm which step matters.

**E12.** Write `drop_one_feature_report(model, X, y)` that refits with each feature removed in turn and
reports the drop in cross-validated score. Run it on 04-01's leaky churn features and confirm it
identifies `months_on_file`.

### Interpretation

**E13.** A colleague's model uses `number_of_customer_service_calls` to predict churn and scores well.
Give the argument that this is legitimate and the argument that it is leakage, and say what single fact
would settle it.

**E14.** A medical model predicting a diagnosis uses `hospital_department` as a feature and performs
excellently. Explain the likely mechanism, why it may still be useful, and why it would probably fail if
deployed at a different hospital.

### Debugging

**E15.** Your pipeline is correct, the split is grouped and chronological, and the model still scores far
too well. Name three remaining places the answer could be entering.

**E16.** A model's score drops from 0.94 to 0.71 when you move preprocessing inside the pipeline. A
colleague asks which number to report. Answer, and then say what you would investigate next - because
0.71 is not necessarily the end of the story either.

### Exam and interview reasoning

**E17.** "What is data leakage and how do you prevent it?" Answer in 90 seconds, with one concrete number.
Then handle: "we cross-validate everything, so we are covered, right?"

### Transfer to a different situation

**E18.** You are predicting whether a job applicant will be hired, from their CV. List four plausible
leaks specific to this problem, one of each kind, and say how you would check for each.

### Explain it to someone non-technical

**E19.** Explain leakage to a manager in under 90 words, using an analogy, and include why the problem is
usually invisible until deployment.

### Optional challenge

**E20.** Build a **leakage detector** that does not require you to know what the leak is: for each
feature, compute the cross-validated score of a model using **only that feature**, and flag any single
feature that alone achieves close to the full model's score. Run it on 04-01's leaky churn table and on
this chapter's noise data. What does it catch, what does it miss, and why is "one feature is nearly as
good as all of them" a suspicious pattern rather than a conclusive one?

In [ ]:
# Your workspace. In memory: make_data, preprocessing_experiment, differences,
# noise, coin_flip, leaky_scores, honest_scores, folds, width_table,
# category, label, leaky, honest_folds, with_duplicates, duplicate_labels,
# original_row_id, neighbour_score, measured.

## Mastery check

- [ ] Name the four kinds and the mechanism of each
- [ ] State the severity principle and use it to classify a step you have not seen before
- [ ] Explain why a scaler is nearly harmless and a selector is not
- [ ] Produce a large inflation from pure noise, deliberately, to prove you understand it
- [ ] Apply the three-symptom checklist to somebody else's result
- [ ] Say what a pipeline makes impossible, and what it does not cover
- [ ] Write the three assertions that catch the structural leaks

## What should now feel instinctive

- Asking "did this step look at `y`?" of every transformation
- Checking `duplicated().sum()` before anything else
- Treating a wonderful score as a symptom, and having an expectation before you look
- Noticing when folds agree too well
- Putting every fitted step inside the pipeline, on the cheap-insurance argument rather than the
  frightening-story one

## Flashcards

| Front | Back |
|---|---|
| Leakage | Information reaching the model that will not exist at prediction time |
| The four kinds | Target, duplicate/group, temporal, preprocessing |
| Severity principle | Damage is proportional to how much the leaked step learned about `y` |
| Scaler fitted on everything | +0.0005. Two numbers per column, and it never sees `y` |
| Feature selection on everything | 79.5% accuracy on a coin flip with 5,000 noise columns |
| Target encoding on everything | AUC 0.7659 on a coin flip with 149 categories over 600 rows |
| Why high cardinality is worse | 4 rows per category means a row is a quarter of its own encoding |
| Duplicate rows | +0.1326 here. The test row is a training example at distance zero |
| Symptom 1 | A score better than the problem allows |
| Symptom 2 | One feature dominates; drop it and the model collapses |
| Symptom 3 | Folds agree too well - leakage raises the mean and lowers the variance |
| The structural fix | Every fitted step inside a `Pipeline`, plus three assertions |
| What a pipeline does not fix | Target leakage, group leakage and temporal leakage - only the fourth kind |

## Next

**04-06 · Preprocessing: imputation, encoding, scaling, transforms.** This chapter treated preprocessing
as a hazard. The next one treats it as a subject: what each transformation is actually for, what it
assumes, and how to choose between them - imputing a missing value, encoding a category with fifty levels,
deciding whether to log a skewed column.

The leakage discipline established here carries forward as a constraint on all of it: **every one of
those steps is fitted, every fitted step is fitted on training data only, and 04-07 makes that automatic.**